# AI Engineer Challenge - Dot Indonesia
## Object Tracking Mobil dengan YOLO + Teori AI

**Kandidat:** Gymnastiar Al K

---
Notebook ini berisi implementasi object tracking menggunakan YOLO (Bagian 1A)
serta jawaban untuk soal teori (Bagian 2).

---
# Bagian 1A: Object Tracking Mobil dengan Pre-trained Model YOLO

Pipeline:
1. Install dependencies
2. Load pre-trained YOLO model (YOLOv8)
3. Download sample video
4. Deteksi + tracking kendaraan per frame
5. Visualisasi hasil

In [ ]:
# Install dependencies
# ultralytics untuk YOLO, opencv untuk video processing
!pip install -q ultralytics opencv-python-headless matplotlib numpy tqdm
print("dependencies installed")

In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from IPython.display import display, Video
from tqdm.notebook import tqdm
import os
from pathlib import Path

print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

### Load Pre-trained YOLO

Menggunakan YOLOv8n (nano) — versi paling ringan, cocok untuk running di Colab.
Model sudah pre-trained di dataset COCO yang memiliki class kendaraan seperti car, bus, truck, motorcycle.

In [ ]:
model = YOLO('yolov8n.pt')

# Kelas kendaraan di COCO: 2=car, 3=motorcycle, 5=bus, 7=truck
vehicle_classes = [2, 3, 5, 7]

print(f"Model: yolov8n.pt")
print(f"Target classes: {[model.names[c] for c in vehicle_classes]}")

### Download Sample Video

Menggunakan sample video dari sumber publik. Disarankan upload video sendiri
atau gunakan link video lalu lintas dari repositori open source.

In [ ]:
# Download sample traffic video
# Ganti URL ini dengan video lalu lintas pilihan sendiri kalo mau
VIDEO_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/traffic.mp4"
VIDEO_PATH = 'traffic.mp4'

!wget -q --show-progress -O $VIDEO_PATH $VIDEO_URL || \
    echo "Link utama tidak tersedia, upload video sendiri via Colab."

if os.path.exists(VIDEO_PATH) and os.path.getsize(VIDEO_PATH) > 10000:
    print(f"Video downloaded: {os.path.getsize(VIDEO_PATH)/1024:.1f} KB")
else:
    print("\nVideo tidak terdownload. Upload manual:")
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = list(uploaded.keys())[0]
        print(f"Using uploaded file: {VIDEO_PATH}")

### Deteksi dan Tracking

Fungsi ini melakukan deteksi kendaraan menggunakan YOLO + ByteTrack.
ByteTrack mempertahankan ID objek antar frame sehingga tracking bersifat konsisten.

In [ ]:
def detect_and_track(frame, model, conf_threshold=0.4):
    """Deteksi kendaraan dan tracking dengan ByteTrack."""
    results = model.track(frame, persist=True, conf=conf_threshold, iou=0.5,
                         tracker='bytetrack.yaml', verbose=False)

    detections = []
    frame_out = frame.copy()

    if results[0].boxes is None:
        return frame_out, detections

    boxes = results[0].boxes.xyxy.cpu().numpy()
    confs = results[0].boxes.conf.cpu().numpy()
    cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
    track_ids = results[0].boxes.id.cpu().numpy().astype(int) if results[0].boxes.id is not None else [-1]*len(boxes)

    for box, conf, cls_id, tid in zip(boxes, confs, cls_ids, track_ids):
        if cls_id not in vehicle_classes:
            continue

        x1, y1, x2, y2 = map(int, box)
        label = f"{'ID:'+str(tid)+' ' if tid > 0 else ''}{model.names[cls_id]} {conf:.2f}"

        detections.append({
            'bbox': (x1, y1, x2, y2),
            'conf': float(conf),
            'class': model.names[cls_id],
            'track_id': int(tid)
        })

        # Warna berbeda per track ID
        color = (tid * 127 % 255, tid * 63 % 255, tid * 255 % 255) if tid > 0 else (0, 255, 0)
        cv2.rectangle(frame_out, (x1, y1), (x2, y2), color, 2)

        # Label background
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
        cv2.rectangle(frame_out, (x1, y1-20), (x1+tw+5, y1), color, -1)
        cv2.putText(frame_out, label, (x1+2, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    return frame_out, detections

print("Function ready")

### Proses Video

Iterasi setiap frame, jalankan deteksi + tracking, lalu simpan sebagai video baru.

In [ ]:
def process_video(video_path, model, output_name='output_tracked.mp4', conf=0.4, max_frames=300):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total = min(total, max_frames)

    out = cv2.VideoWriter(output_name, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    vehicle_counts = []

    print(f"Processing {total} frames ({fps} fps, {w}x{h})...")
    for i in tqdm(range(total)):
        ret, frame = cap.read()
        if not ret:
            break

        annotated, dets = detect_and_track(frame, model, conf)
        cv2.putText(annotated, f"Frame {i+1}/{total} | Vehicles: {len(dets)}",
                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
        out.write(annotated)
        vehicle_counts.append(len(dets))

    cap.release()
    out.release()

    print(f"\nDone. Output: {output_name}")
    print(f"Average vehicles/frame: {np.mean(vehicle_counts):.1f}")
    print(f"Max vehicles in a frame: {np.max(vehicle_counts)}")
    return output_name, vehicle_counts


out_video, counts = process_video(VIDEO_PATH, model, 'output_tracked.mp4', 0.4, 300)

In [ ]:
# Compress for Colab display
!ffmpeg -y -i output_tracked.mp4 -vcodec libx264 -crf 23 output_compressed.mp4 -loglevel error 2>/dev/null

if os.path.exists('output_compressed.mp4'):
    display(Video('output_compressed.mp4', width=720))
elif os.path.exists('output_tracked.mp4'):
    display(Video('output_tracked.mp4', width=720))
else:
    print("Output video not found, pastikan video berhasil diproses.")

### Sample Frame Results

Visualisasi beberapa frame untuk melihat detail deteksi.

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
total_fr = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
# Ambil 6 sample frame
indices = np.linspace(0, min(total_fr-1, 300), 6, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Detection Results - Sample Frames', fontsize=14)

for ax, idx in zip(axes.flatten(), indices):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ret, frame = cap.read()
    if not ret:
        continue
    annotated, dets = detect_and_track(frame, model)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {idx} | {len(dets)} vehicles", fontsize=11)
    ax.axis('off')

cap.release()
plt.tight_layout()
plt.show()

### Tracking Summary

In [ ]:
print(f"Total frames processed: {len(counts)}")
print(f"Total vehicle detections: {np.sum(counts)}")
print(f"Average per frame: {np.mean(counts):.1f}")
print(f"Max vehicles in a single frame: {np.max(counts)}")
print(f"Model: yolov8n.pt | Tracker: ByteTrack | Confidence threshold: 0.4")

# Distribution plot
plt.figure(figsize=(10, 3))
plt.plot(counts, alpha=0.7, linewidth=1)
plt.axhline(np.mean(counts), color='r', linestyle='--', label=f'Mean: {np.mean(counts):.1f}')
plt.xlabel('Frame')
plt.ylabel('Vehicle Count')
plt.legend()
plt.title('Vehicle Count Distribution Across Frames')
plt.show()

---
# Bagian 2: Teori Artificial Intelligence
---

### Soal 2a: Apa yang dimaksud dengan Artificial Intelligence (AI)?

Artificial Intelligence (AI) adalah cabang ilmu komputer yang berfokus pada pengembangan sistem yang mampu melakukan tugas yang biasanya membutuhkan kecerdasan manusia. AI bekerja dengan memproses data, mengenali pola, dan membuat keputusan atau prediksi berdasarkan pola tersebut.

**Dua contoh AI dalam kehidupan sehari-hari:**

1. **Sistem rekomendasi (Netflix, YouTube, Spotify)** — AI menganalisis riwayat interaksi pengguna untuk merekomendasikan konten yang relevan, menggunakan teknik collaborative filtering dan deep learning.

2. **Asisten virtual (Siri, Google Assistant, ChatGPT)** — Menggunakan Natural Language Processing (NLP) untuk memahami perintah suara atau teks, lalu memberikan respons yang sesuai.

### Soal 2b: Perbedaan Supervised Learning dan Unsupervised Learning

| Aspek | Supervised Learning | Unsupervised Learning |
|-------|--------------------|-----------------------|
| Data | Berlabel (labeled) | Tidak berlabel (unlabeled) |
| Tujuan | Memprediksi output dari input | Menemukan struktur/kelompok dalam data |
| Feedback | Ada (dari ground truth) | Tidak ada |
| Contoh algoritma | Linear Regression, Random Forest, SVM | K-Means, DBSCAN, PCA |

**Contoh Supervised Learning:** Klasifikasi email spam/not-spam. Model dilatih dengan ribuan email yang sudah dilabeli, sehingga bisa memprediksi email baru masuk ke kategori mana.

**Contoh Unsupervised Learning:** Segmentasi pelanggan berdasarkan pola pembelian. Model mengelompokkan pelanggan ke dalam cluster tanpa label sebelumnya — misalnya cluster "pembeli reguler", "pembeli musiman", dll.

### Soal 3a: Apa itu Feature dalam Machine Learning?

Feature adalah variabel atau atribut yang dapat diukur dari data dan digunakan sebagai input model machine learning. Feature membantu model memahami pola dalam data untuk membuat prediksi.

**Contoh:** Untuk model prediksi harga rumah, feature dapat berupa luas tanah, jumlah kamar, lokasi, tahun dibangun, dan jarak ke pusat kota.

**Mengapa pemilihan feature penting?**

1. **Relevansi** — Feature yang tidak relevan menambah noise dan menurunkan akurasi.
2. **Curse of Dimensionality** — Semakin banyak feature, semakin banyak data yang dibutuhkan. Feature yang tidak informatif hanya memperbesar dimensi tanpa kontribusi.
3. **Interpretability** — Model dengan feature yang sedikit dan bermakna lebih mudah dijelaskan.
4. **Efisiensi komputasi** — Feature yang tepat membuat training lebih cepat.
5. **Overfitting** — Feature berlebihan dapat menyebabkan model menghafal noise.

Teknik seleksi feature yang umum: feature importance (dari tree-based models), PCA, mutual information, dan Recursive Feature Elimination (RFE).

### Soal 3b: Apa itu Fine-tuning dalam Machine Learning?

Fine-tuning adalah teknik transfer learning di mana model yang sudah pre-trained pada dataset besar dan umum dilatih ulang secara spesifik pada dataset yang lebih kecil untuk tugas tertentu.

Prosesnya:
- Ambil model pre-trained (misal: YOLO, ResNet, BERT)
- Ganti layer akhir (head) sesuai tugas baru
- Latih ulang dengan learning rate kecil (cukup beberapa epoch karena model sudah memiliki pengetahuan dasar)

**Contoh kasus: Deteksi objek khusus — deteksi sampah pesisir**

YOLO yang pre-trained di COCO (80 kelas umum) dapat di-fine-tune untuk mendeteksi sampah plastik di wilayah pesisir — kelas yang tidak ada di COCO. Karena model sudah memahami konsep dasar objek (tepi, tekstur, bentuk), fine-tuning hanya membutuhkan ratusan gambar berlabel, bukan ribuan. Ini jauh lebih efisien dibandingkan training dari scratch.

**Manfaat fine-tuning:**
- Konvergensi lebih cepat (epoch lebih sedikit)
- Membutuhkan data lebih sedikit
- Biaya komputasi lebih rendah
- Performa lebih baik pada dataset kecil dibandingkan training dari nol